# A Global Mini Integrated Assessment Model: Energy Transitions under a Carbon Budget

**ENGG562/ENV462 · Climate Change Governance: Science, Data & Models · LUMS · Fall 2026**

**Module 2: Scenarios & Integrated Assessment** · Coding exercise 05 · Mini integrated assessment model series, notebook 3 of 5 · Requires coding exercise 04

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LUMS-WIT/climate-futures/blob/main/content-src/coding/05-global-mini-iam.ipynb)

Notebook 04 built the MESSAGE formulation for Westeros. This notebook applies the **same formulation**, with the same commands, to a stylised model of the **global** energy system from 2020 to 2100. The model is small enough to solve in a second, yet large enough to reproduce the main questions studied with the IAM scenarios of the IPCC Sixth Assessment Report (coding exercise 02): how fast emissions must fall to stay within a carbon budget, which technologies carry the transition, how high the carbon price must be, and what role negative emissions play.

| Notebook | Content |
|---|---|
| 03 | Linear optimisation, shadow prices, and a small capacity expansion model |
| 04 | Westeros: the MESSAGE formulation on a small example |
| **05 (this notebook)** | **World: the same formulation applied to a global energy system** |
| 06 | MESSAGE-MACRO: linking the energy model to a macro-economic growth model |
| 07 | From the mini model to the real `message_ix` package |

> **Important.** The input data of this model are rounded, illustrative values of a plausible order of magnitude (see Part 1). The model is a teaching tool for understanding how integrated assessment models work. Its results are not projections and should not be cited as such; for that purpose, the AR6 scenario ensemble of coding exercise 02 is the appropriate source.

Prompts: **💬 Predict**, **✏️ Exercise** (replace every `...` or `___`; self-checks print ✅ or ❌), and **💬 Question**.

## Learning objectives

On completing this notebook, students should be able to:

1. Describe the reference energy system of a global integrated assessment model and relate each element to its Westeros counterpart.
2. Check the base-year calibration of a model against observed data.
3. Translate a remaining carbon budget from the IPCC into a model constraint, and estimate the warming of a pathway with the transient climate response to cumulative emissions (TCRE).
4. Compare carbon-budget and carbon-tax scenarios in terms of emissions, technologies, carbon prices, and costs.
5. Run sensitivity analyses on contested assumptions (availability of carbon capture, bioenergy, nuclear power, the pace of solar expansion) and explain their effect.

## Structure of the notebook

| Part | Content | Approximate time |
|---|---|---|
| 0 | Setup | 5 min |
| 1 | The global reference energy system | 15 min |
| 2 | Checking the base year | 10 min |
| 3 | The baseline scenario | 20 min |
| 4 | Carbon budgets for 2 °C and 1.5 °C | 25 min |
| 5 | A carbon tax | 15 min |
| 6 | In-class activity: contested assumptions | 45 min |
| 7 | Homework assignment | One week |
| 8 | Limitations and references | — |

---
# Part 0 · Setup

In [ ]:
import sys

if "google.colab" in sys.modules:
    !pip install --quiet pyomo highspy
    !wget --quiet --timestamping https://raw.githubusercontent.com/LUMS-WIT/climate-futures/main/content-src/coding/miniiam.py

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import miniiam as mi
from miniiam import make_df

plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
pd.set_option("display.precision", 1)
pd.set_option("display.width", 160)

GWA_TO_PWH = 8.76 / 1000   # 1 GWa = 8.76 TWh
GWA_TO_EJ = 0.031536       # 1 GWa = 0.031536 EJ

In [ ]:
def check(description, condition):
    """Self-check helper: prints a tick or a cross. It never stops the notebook."""
    try:
        passed = bool(condition())
    except Exception:
        passed = False
    print(("✅ " if passed else "❌ ") + description)

---
# Part 1 · The global reference energy system

The world model has the same four levels as Westeros, with one more at the start of the chain: **primary** energy (fuels as extracted), **secondary** energy (electricity from power plants, refined oil products), **final** energy (delivered to users), and **useful** energy (the service). Three useful-energy demands drive the model:

- `spec`: **specific electricity uses** that only electricity can supply (appliances, motors, lighting, information technology). This is the "light" of Westeros at a global scale.
- `therm`: **thermal energy** for industry and buildings (process heat, space and water heating, cooking).
- `transport`: **transport services**, expressed as useful energy delivered by vehicles.

```
 PRIMARY              SECONDARY                    FINAL                      USEFUL
 coal_extr ─ coal ──┬─ coal_ppl, coal_ccs ─┐
 gas_extr ── gas ───┼─ gas_ppl, gas_ccs ───┤                    ┌─ elec_spec ─────────▶ spec
 oil_extr ─ crude ─ oil_ref ─ oil ─────────┼─────────────┐      ├─ elec_i, h2_i ──────▶ therm  ◀─ coal_i, gas_i, oil_i, bio_i
 bio_extr ── biomass┼─ bio_ppl, bio_ccs ───┤ electricity ─ elec_t_d ─ electricity ─┤
                    nuc_ppl, hydro_ppl ────┤                    ├─ elec_trp, h2_trp ──▶ transport ◀─ oil_trp, gas_trp, bio_trp
                    wind_ppl, solar_pv ────┘                    └─ h2_elec ─ hydrogen
```

The model is built by the function `mi.build_world()`, which uses only the commands of notebook 04 (`add_horizon`, `add_set`, `make_df`, `add_par`). Its data are stored in readable tables in the module.

In [ ]:
base = mi.build_world(mi.Scenario(model="World", scenario="baseline"))

tec = pd.DataFrame(mi.WORLD_TECH, index=["input", "input level", "output", "output level", "efficiency"]).T
tec

In [ ]:
cap = pd.DataFrame({t: {"inv_cost 2030 (USD/kW)": mi._interp(d["inv"], 2030),
                        "inv_cost 2050 (USD/kW)": mi._interp(d["inv"], 2050),
                        "fix_cost (USD/kW/yr)": d["fix"], "var_cost (USD/kWa)": d["var"],
                        "capacity factor": d["cf"], "lifetime (yr)": d["life"]}
                    for t, d in mi.WORLD_CAPACITY.items()}).T
cap

**Where the numbers come from.** All values are rounded, illustrative assumptions of a plausible order of magnitude around 2020. Fuel costs are expressed per kWa: 80 USD/kWa for coal corresponds to about 2.5 USD per GJ, and 300 USD/kWa for crude oil to about 9.5 USD per GJ (roughly 55–60 USD per barrel). CO₂ emission factors of fuels are the IPCC 2006 default values (94.6 kg CO₂ per GJ for coal, 56.1 for natural gas, 73.3 for crude oil). CCS plants capture 90% of the CO₂ in their fuel; for biomass with CCS this gives **net negative** emissions, because the carbon in the biomass was taken from the atmosphere as the plants grew.

Three groups of constraints keep the linear model from producing physically implausible results:

- **Growth constraints** (`growth_activity_up`, `initial_activity_up`): for example, solar and wind output can grow by at most 10% per year, plus a start-up allowance.
- **Decline constraints** (`growth_activity_lo`): existing fossil technologies can shrink by at most 4–6% per year, reflecting the slow turnover of capital stock and infrastructure.
- **Potentials** (`bound_activity_up`): biomass supply is limited to about 120 EJ per year and hydropower to about 7,900 TWh per year. A **relation** (`relation_activity`, named `vre_share`) limits wind and solar to 60% of electricity generation, a crude stand-in for the storage and grid flexibility that a model with one time slice per year cannot represent.

In [ ]:
base.par("growth_activity_up").groupby("technology")["value"].first().to_frame("max. annual growth").T

---
# Part 2 · Checking the base year

Before any scenario is interpreted, a modeller checks that the base year resembles the real world. The model is given the activity of each technology in 2020 (`historical_activity`). From it, total electricity generation, final energy, and CO₂ emissions in 2020 can be computed and compared with observed data.

Observed reference values: global electricity generation in 2020 was about 26,800 TWh, of which 7,593 TWh renewable (IEA, *World Energy Outlook 2021*); total final consumption was of the order of 385–400 EJ; and fossil CO₂ emissions, including cement production, were 34.8 Gt (Global Carbon Budget 2021).

### ✏️ Exercise 1: Base-year emissions

Compute the CO₂ emissions implied by the 2020 activity. Each technology that burns a fossil fuel uses `activity / efficiency` units of fuel, and each unit of fuel emits the factor in `mi.EF` (Mt CO₂ per GWa). Oil products pass through the refinery (`oil_ref`, efficiency 0.9) before use, so their crude-oil use is a further factor 1/0.9 larger. Store the total in Gt as `co2_2020`.

In [ ]:
hist = mi.WORLD_HIST_2020
fuel_use = {"coal": 0.0, "gas": 0.0, "oil": 0.0}
for t, act in hist.items():
    inp, _, _, _, eff = mi.WORLD_TECH[t]
    if inp in ("coal", "gas"):
        fuel_use[inp] += act / eff
    elif inp == "oil":
        fuel_use["oil"] += act / eff / ___

co2_2020 = sum(fuel_use[f] * mi.EF[f] for f in fuel_use) / ___
print(f"Implied CO2 emissions in 2020: {co2_2020:.1f} Gt")

In [ ]:
check("about 30 Gt CO2", lambda: 28 < co2_2020 < 33)

In [ ]:
gen = [t for t, v in mi.WORLD_TECH.items() if v[2] == "electricity" and v[3] == "secondary"]
elec_2020 = sum(mi.WORLD_HIST_2020.get(t, 0) for t in gen) * GWA_TO_PWH * 1000
renew_2020 = sum(mi.WORLD_HIST_2020.get(t, 0) for t in ["hydro_ppl", "wind_ppl", "solar_pv", "bio_ppl"]) * GWA_TO_PWH * 1000
print(f"Electricity generation 2020: {elec_2020:,.0f} TWh, of which renewable {renew_2020:,.0f} TWh")

**💬 Question.** The model's CO₂ emissions in 2020 are lower than the 34.8 Gt reported by the Global Carbon Budget. List at least two sources of CO₂ that the model does not include, and state whether the omission matters for the questions studied in this notebook.

*Answer:* *(write the answer here)*

---
# Part 3 · The baseline scenario

The baseline has no climate policy. Demand for useful energy grows along an SSP2-like path ("middle of the road", Lecture 8): specific electricity uses grow by 2.5% per year to 2050, transport by 2%, and thermal energy by 1%.

In [ ]:
base.solve()
print(f"Total discounted system cost: {base.var('OBJ') / 1e6:.1f} trillion USD")

In [ ]:
COLORS = {"coal_ppl": "#4d4d4d", "coal_ccs": "#8c8c8c", "gas_ppl": "#e08214", "gas_ccs": "#fdb863",
          "nuc_ppl": "#8073ac", "hydro_ppl": "#4393c3", "wind_ppl": "#5aae61", "solar_pv": "#fee08b",
          "bio_ppl": "#a6611a", "bio_ccs": "#dfc27d"}


def plot_power(s, ax=None, title=None):
    act = mi.timeseries(s, "ACT")[gen] * GWA_TO_PWH
    ax = ax or plt.gca()
    act.plot.area(ax=ax, color=[COLORS[t] for t in gen], linewidth=0, legend=False)
    ax.set(title=title or f"Electricity generation: {s.scenario}", ylabel="PWh per year", xlabel="")
    return ax


def emissions(s):
    return s.var("EMISS").set_index("year")["lvl"] / 1000   # Gt CO2 per year


fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
plot_power(base, axes[0])
axes[0].legend(loc="upper left", fontsize=8, ncol=2)
emissions(base).plot(ax=axes[1], marker="o", color="black", title="CO2 emissions (Gt per year)")
axes[1].axhline(0, color="grey", lw=0.5)
plt.tight_layout()
plt.show()

The end-use technologies show how demand is met in each sector.

In [ ]:
END_USE = {"therm": ["coal_i", "gas_i", "oil_i", "bio_i", "elec_i", "h2_i"],
           "transport": ["oil_trp", "gas_trp", "bio_trp", "elec_trp", "h2_trp"]}

def plot_end_use(s):
    act = mi.timeseries(s, "ACT")
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for ax, (sector, techs) in zip(axes, END_USE.items()):
        act[techs].plot.area(ax=ax, linewidth=0, title=f"{sector}: useful energy by technology (GWa)")
        ax.legend(fontsize=8, loc="upper left")
    plt.tight_layout()
    plt.show()

plot_end_use(base)

**💬 Question.** Describe the baseline in three sentences: which energy source dominates electricity, how transport changes, and what happens to emissions. Then identify one feature of the result that is a consequence of the model being **linear** rather than a realistic expectation.

*Answer:* *(write the answer here)*

### ✏️ Exercise 2: What the constraints do

Remove all growth and decline constraints from a clone of the baseline (`remove_par("growth_activity_up")`, and the same for `initial_activity_up` and `growth_activity_lo`), solve it as `free`, and plot its end-use technologies with `plot_end_use(free)`. Compute `coal_i_2030`, the activity of `coal_i` in 2030 in the unconstrained run.

In [ ]:
free = base.clone(scenario="no dynamic constraints")
for name in [___, ___, ___]:
    free.remove_par(name)
free.solve()
plot_end_use(free)
coal_i_2030 = mi.timeseries(free, "ACT").loc[2030, ___]
print(f"coal for heat in 2030: {coal_i_2030:,.0f} GWa (baseline: {mi.timeseries(base, 'ACT').loc[2030, 'coal_i']:,.0f})")

In [ ]:
check("without dynamic constraints, coal takes over thermal demand in 2030",
      lambda: coal_i_2030 > 2 * mi.timeseries(base, "ACT").loc[2030, "coal_i"])

**💬 Question.** Without dynamic constraints, the model replaces almost all gas and oil boilers with coal within ten years. Why is this unrealistic, and what does the exercise reveal about how results of large IAMs depend on such constraints?

*Answer:* *(write the answer here)*

---
# Part 4 · Carbon budgets for 2 °C and 1.5 °C

Global warming is roughly proportional to cumulative CO₂ emissions. The constant of proportionality is the **transient climate response to cumulative emissions (TCRE)**, which the IPCC (AR6 WGI) assesses at 0.45 °C per 1,000 Gt CO₂ (likely range 0.27–0.63). The IPCC derives from it **remaining carbon budgets** from the start of 2020:

| Target | Remaining budget from 2020 |
|---|---|
| 1.5 °C with 50% probability | 500 Gt CO₂ |
| 2 °C with 67% probability | 1,150 Gt CO₂ |

These budgets already account for the expected warming from non-CO₂ gases. As in notebook 04, a `bound_emission` with `type_year="cumulative"` limits the **average** annual emissions over the model horizon, so a budget is entered as its total divided by the number of years in the horizon (2021–2100, 80 years).

In [ ]:
horizon_years = sum(base.par("duration_period").set_index("year").loc[base.model_years(), "value"])


def budget_scenario(budget_gt, name, start=base):
    s = start.clone(scenario=name)
    s.add_par("bound_emission", make_df("bound_emission", node="World", type_emission="GHG", type_tec="all",
                                        type_year="cumulative", value=budget_gt * 1000 / horizon_years,
                                        unit="MtCO2"))
    s.solve()
    return s


print("horizon length:", horizon_years, "years")
b2c = budget_scenario(1150, "2C (1150 Gt)")

**💬 Predict before running the next cell.** How will the 2 °C budget change the electricity mix compared with the baseline, and roughly when will global CO₂ emissions reach net zero?

*Answer:* *(write the answer here)*

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
plot_power(base, axes[0])
plot_power(b2c, axes[1])
axes[1].legend(loc="upper left", fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

### ✏️ Exercise 3: The 1.5 °C budget

Create the 1.5 °C scenario `b15` with the function `budget_scenario`. Then build a table `summary` with one row per scenario (baseline, 2 °C, 1.5 °C) and the columns: emissions in 2030 and 2050 (Gt), the first year with net-zero or negative emissions, cumulative emissions 2021–2100 (Gt), and peak warming (°C, from `mi.warming`).

In [ ]:
b15 = budget_scenario(___, "1.5C (500 Gt)")

def summarise(s):
    e = emissions(s)
    net_zero = e[e <= 0].index.min()
    return {"CO2 2030 (Gt)": e[2030], "CO2 2050 (Gt)": e[___],
            "net zero": net_zero,
            "cumulative 2021-2100 (Gt)": mi.cumulative_co2(s).iloc[-1],
            "peak warming (°C)": mi.warming(s).___()}

summary = pd.DataFrame({s.scenario: summarise(s) for s in [base, b2c, b15]}).T
summary

In [ ]:
check("the 1.5 °C scenario stays within 500 Gt", lambda: summary.loc["1.5C (500 Gt)", "cumulative 2021-2100 (Gt)"] <= 500.5)
check("the 1.5 °C scenario cuts 2050 emissions more than the 2 °C scenario",
      lambda: summary.loc["1.5C (500 Gt)", "CO2 2050 (Gt)"] < summary.loc["2C (1150 Gt)", "CO2 2050 (Gt)"])

`mi.warming()` adds the CO₂-induced warming (TCRE × cumulative emissions since 2020) to the observed warming of about 1.1 °C. It ignores non-CO₂ gases and is therefore a lower estimate; it is useful for comparing scenarios, not for stating whether a target is met.

For comparison, in the AR6 scenarios that limit warming to 1.5 °C with no or limited overshoot (category C1), global net CO₂ emissions reach zero in the early 2050s; in the scenarios that limit warming to 2 °C with 67% probability (C3), in the early 2070s (IPCC AR6 WGIII, Summary for Policymakers, C.1).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for s, c in [(base, "black"), (b2c, "tab:blue"), (b15, "tab:green")]:
    emissions(s).plot(ax=axes[0], marker="o", color=c, label=s.scenario)
    p = s.var("PRICE_EMISSION")
    if len(p):
        p.set_index("year")["lvl"].plot(ax=axes[1], marker="o", color=c, label=s.scenario, logy=True)
axes[0].axhline(0, color="grey", lw=0.5)
axes[0].set(title="CO2 emissions (Gt per year)")
axes[1].set(title="Carbon price (USD per t CO2, log scale)")
for ax in axes:
    ax.legend()
plt.tight_layout()
plt.show()

**💬 Question.** Compare the timing of net zero in the model's 1.5 °C scenario with the AR6 C1 scenarios, and compare the model's carbon price in 2030 with the price that the shadow price of BECCS implies in 2100. Explain the connection between the two observations.

*Answer:* *(write the answer here)*

---
# Part 5 · A carbon tax

Instead of a budget, a government can set a carbon price. MESSAGE implements it with `tax_emission`. A common design is a price that starts at a moderate level and rises at a fixed rate.

In [ ]:
years = base.model_years()
tax_path = [30 * 1.05 ** (y - 2030) for y in years]   # 30 USD/t in 2030, rising by 5% per year

tax = base.clone(scenario="tax 30 USD/t +5%/yr")
tax.add_par("tax_emission", make_df("tax_emission", node="World", type_emission="GHG", type_tec="all",
                                    type_year=years, value=tax_path, unit="USD/tCO2"))
tax.solve()
pd.Series(tax_path, index=years, name="tax (USD/t)").round(0).to_frame().T

In [ ]:
summary_tax = pd.DataFrame({s.scenario: summarise(s) for s in [base, b2c, b15, tax]}).T
summary_tax

**💬 Question.** Compare the cumulative emissions of the tax scenario with the two budget scenarios. A government that sets a carbon tax does not know in advance which budget it will achieve. What does this imply for the choice between a tax and a cap (notebook 03, Weitzman 1974) when the costs of technologies are uncertain?

*Answer:* *(write the answer here)*

### Cost of the transition

The objective value measures the total discounted cost of the energy system. Comparing it across scenarios gives the **mitigation cost**, which must exclude tax payments (they are transfers). `COST_NODAL_NET` reports the annual system cost without taxes.

In [ ]:
def system_cost(s):
    return s.var("COST_NODAL_NET").set_index("year")["lvl"] / 1e3   # billion USD per year

cost = pd.DataFrame({s.scenario: system_cost(s) for s in [base, b2c, b15, tax]})
(cost.sub(cost[base.scenario], axis=0)).drop(columns=base.scenario).round(0)

**💬 Question.** The table shows the additional annual energy-system cost of each scenario relative to the baseline, in billion USD. Why is this not the same as the cost of climate policy to the economy, and which model in this series addresses the difference?

*Answer:* *(write the answer here)*

---
# Part 6 · In-class activity: contested assumptions

Every integrated assessment result rests on assumptions that experts disagree about. In this activity, groups test how the 2 °C scenario changes when one such assumption changes. The cell below provides a helper that applies a change to a clone of the baseline and then imposes the 2 °C budget.

In [ ]:
def variant(name, change, budget_gt=1150):
    s = base.clone(scenario=name)
    change(s)
    try:
        return budget_scenario(budget_gt, name, start=s)
    except RuntimeError as err:
        print(f"{name}: no feasible solution. {err}")
        return None


def zero_bound(techs):
    def change(s):
        for t in techs:
            s.add_par("bound_activity_up", make_df("bound_activity_up", node_loc="World", technology=t,
                                                   year_act=years, mode="standard", time="year",
                                                   value=0.0, unit="GWa"))
    return change

## 6.1 Core task (all groups)

### ✏️ Exercise 4: A world without carbon capture

CCS is not yet deployed at scale, and some argue that it never will be. Create the variant `no_ccs`, in which `coal_ccs`, `gas_ccs`, and `bio_ccs` cannot be used, using `variant` and `zero_bound`. Add it to the comparison table and plot its electricity mix.

In [ ]:
no_ccs = variant("2C without CCS", zero_bound([___, ___, ___]))
comparison = pd.DataFrame({s.scenario: summarise(s) for s in [b2c, no_ccs]}).T
comparison["carbon price 2030"] = [s.var("PRICE_EMISSION").set_index("year").loc[2030, "lvl"] for s in [b2c, no_ccs]]
print(comparison)
plot_power(no_ccs)
plt.show()

In [ ]:
check("without CCS, emissions in 2050 are lower", lambda: emissions(no_ccs)[2050] < emissions(b2c)[2050])
check("without CCS, the 2030 carbon price is higher",
      lambda: comparison.loc["2C without CCS", "carbon price 2030"] > comparison.loc["2C (1150 Gt)", "carbon price 2030"])

**💬 Question.** Explain why removing CCS **raises** the near-term carbon price and **lowers** emissions in 2050, even though the budget is unchanged.

*Answer:* *(write the answer here)*

## 6.2 ENGG track: the power sector

### ✏️ Exercise 5: Nuclear phase-out and the pace of solar

**(a)** Create a variant `no_new_nuclear` in which nuclear activity cannot exceed its 2020 level of 310 GWa (use `bound_activity_up` with value 310 in all years). **(b)** Create a variant `fast_solar` in which the maximum growth rate of `solar_pv` and `wind_ppl` is raised from 10% to 15% per year. Compare both with `b2c` in terms of the carbon price in 2030 and 2050, and the share of solar in electricity in 2050.

In [ ]:
def cap_nuclear(s):
    s.add_par("bound_activity_up", make_df("bound_activity_up", node_loc="World", technology="nuc_ppl",
                                           year_act=years, mode="standard", time="year", value=___, unit="GWa"))

def faster_vre(s):
    for t in ["solar_pv", "wind_ppl"]:
        s.add_par("growth_activity_up", make_df("growth_activity_up", node_loc="World", technology=t,
                                                year_act=years, time="year", value=___, unit="-"))

no_new_nuclear = variant("2C nuclear capped", cap_nuclear)
fast_solar = variant("2C fast VRE", faster_vre)

def solar_share(s, y=2050):
    a = mi.timeseries(s, "ACT").loc[y, gen]
    return a["solar_pv"] / a.sum()

engg = pd.DataFrame({s.scenario: {"price 2030": s.var("PRICE_EMISSION").set_index("year").loc[2030, "lvl"],
                                  "price 2050": s.var("PRICE_EMISSION").set_index("year").loc[2050, "lvl"],
                                  "solar share 2050": solar_share(s)}
                     for s in [b2c, no_new_nuclear, fast_solar]}).T
engg.round(2)

In [ ]:
check("capping nuclear raises the carbon price", lambda: engg.loc["2C nuclear capped", "price 2050"] >= engg.loc["2C (1150 Gt)", "price 2050"])
check("faster VRE growth changes the carbon price by less than 2%",
      lambda: abs(engg.loc["2C fast VRE", "price 2050"] / engg.loc["2C (1150 Gt)", "price 2050"] - 1) < 0.02)

**💬 Question.** Capping nuclear power raises the carbon price, but faster growth of solar and wind hardly changes it. Use the constraints described in Part 1 to explain why, and suggest which additional change would let faster solar growth lower the cost of the 2 °C target.

*Answer:* *(write the answer here)*

## 6.3 ENV track: bioenergy and land

### ✏️ Exercise 6: How much biomass is sustainable?

Global bioenergy potentials are highly uncertain because biomass competes with food production and ecosystems for land. Run the 2 °C scenario with a biomass limit of 1,600 GWa per year (about 50 EJ, roughly the current level) instead of 3,800 GWa. Name the variant `low_bio`, and report its cumulative negative emissions from `bio_ccs` (Gt), its carbon price in 2050, and its net-zero year, compared with `b2c`.

In [ ]:
def less_biomass(s):
    s.add_par("bound_activity_up", make_df("bound_activity_up", node_loc="World", technology="bio_extr",
                                           year_act=years, mode="standard", time="year", value=___, unit="GWa"))

low_bio = variant("2C low biomass", less_biomass)

def beccs_removals(s):
    ef = -mi.CAPTURE * mi.EF["biomass"] / 0.25           # Mt CO2 removed per GWa of electricity
    act = mi.timeseries(s, "ACT")["bio_ccs"]
    return -(act * ef * 10).sum() / 1000                 # Gt over the horizon (10-year periods)

env = pd.DataFrame({s.scenario: {"BECCS removals (Gt)": beccs_removals(s),
                                 "price 2050": s.var("PRICE_EMISSION").set_index("year").loc[2050, "lvl"],
                                 "net zero": summarise(s)["net zero"]}
                    for s in [b2c, low_bio]}).T
env

In [ ]:
check("less biomass raises the carbon price", lambda: env.loc["2C low biomass", "price 2050"] >= env.loc["2C (1150 Gt)", "price 2050"])

**💬 Question.** This model treats biomass as a single commodity with a fixed cost and no land. List three consequences of large-scale bioenergy that a coupled land-use model (such as GLOBIOM, which is linked to MESSAGE in MESSAGEix-GLOBIOM) would capture and this model ignores.

*Answer:* *(write the answer here)*

---
# Part 7 · Homework assignment

**Title:** What does the model need to believe for the 1.5 °C target to be met?

**Deliverables:** (1) the completed notebook, runnable from top to bottom; (2) a brief of no more than 800 words with two to four figures, submitted as a PDF. The due date is announced in class. The assignment is evaluated under the Assignments rubric on the course [Grading page](https://lums-wit.github.io/climate-futures/grading.html).

**Use of AI tools.** In line with the course policy, AI tools may be used for this assignment on the condition that every prompt used is submitted alongside the work, in the final cell of the notebook. AI-assisted and non-AI-assisted submissions are evaluated against different bars.

## Task

Starting from the 1.5 °C scenario (`b15`), identify the assumptions on which its feasibility depends. Run at least **four** variants that change one assumption each (for example: no CCS, low biomass, slower solar growth, a lower share limit for wind and solar, faster decline of coal, higher discount rate), and determine for each whether the 1.5 °C budget can still be met (the model reports an infeasible solution if not), and at what carbon price.

| | **Foundation level** | **Advanced level** |
|---|---|---|
| **ENGG track** | Vary assumptions about power-sector technologies. Report the electricity mix in 2050 and the carbon price for each variant. | As Foundation, and additionally: find the **tightest budget** that remains feasible under each variant (by repeated solving), and present the result as a figure of feasible budget against the assumption. |
| **ENV track** | Vary assumptions about biomass and negative emissions. Report cumulative net-negative emissions and the net-zero year for each variant. | As Foundation, and additionally: compare the model's net-zero year and 2030 and 2050 emissions with the AR6 C1 scenarios from coding exercise 02, and explain the differences with reference to the model's assumptions. |

## Guidance

- Use the `variant()` function of Part 6; the argument `budget_gt=500` applies the 1.5 °C budget.
- An infeasible result is a finding, not an error: it states that under the assumption tested, the target cannot be met with the technologies in the model.
- The brief is written for a reader in a ministry who knows what a carbon budget is but has never run a model. Each figure needs a caption that states what it shows and what follows from it.

In [ ]:
# Homework: begin the analysis here. Add as many cells as needed.

**Prompts used (if any AI tool was used):** *(paste every prompt here, in order)*

---
# Part 8 · Limitations and references

## What this model leaves out

The model has the structure of MESSAGE, but large IAMs such as MESSAGEix-GLOBIOM differ in ways that matter for the results:

- **Regions and trade:** one global region instead of 11 or 12 world regions linked by trade in fuels and electricity. Pakistan and South Asia are not visible.
- **Time slices:** one time step per year instead of seasons and times of day, so the variability of wind and solar is approximated by a share constraint.
- **Vintages:** efficiencies and costs do not vary by the year a plant was built.
- **Land use and non-CO₂ gases:** there is no land model and no methane or nitrous oxide; the climate is represented only by TCRE.
- **Demand:** demand does not respond to prices. Notebook 06 removes this limitation.
- **Data:** all inputs are illustrative.

## References

- IPCC (2021). Summary for Policymakers. In: *Climate Change 2021: The Physical Science Basis.* Contribution of Working Group I to the Sixth Assessment Report of the IPCC. Table SPM.2 (remaining carbon budgets) and Section D.1 (TCRE).
- IPCC (2022). Summary for Policymakers. In: *Climate Change 2022: Mitigation of Climate Change.* Contribution of Working Group III to the Sixth Assessment Report of the IPCC. Section C.1 (timing of net-zero CO₂).
- Anderson, K., and Peters, G. (2016). The trouble with negative emissions. *Science*, 354(6309), 182–183. [doi:10.1126/science.aah4567](https://doi.org/10.1126/science.aah4567)
- Krey, V., Havlík, P., Kishimoto, P. N., Fricko, O., Zilliacus, J., Gidden, M., et al. (2020). *MESSAGEix-GLOBIOM Documentation – 2020 release.* International Institute for Applied Systems Analysis. [doi:10.22022/iacc/03-2021.17115](https://doi.org/10.22022/iacc/03-2021.17115)
- Friedlingstein, P., et al. (2022). Global Carbon Budget 2021. *Earth System Science Data*, 14, 1917–2005. [doi:10.5194/essd-14-1917-2022](https://doi.org/10.5194/essd-14-1917-2022)
- IEA (2021). *World Energy Outlook 2021.* International Energy Agency, Paris.